# Spotify Streaming Audit — Dataset Grammy Awards

## Limpieza y preparación de datos

En esta sección trabajaremos con el dataset adicional de Grammy Awards.

El objetivo es preparar una base limpia, ordenada y reproducible que pueda utilizarse posteriormente para:

- análisis exploratorio,
- cruces con el dataset principal de Spotify,
- SQL,
- visualizaciones,
- dashboards,
- y análisis avanzados del proyecto.

Durante esta limpieza realizaremos:

- carga del archivo,
- revisión inicial,
- limpieza de columnas,
- revisión de nulos,
- eliminación de duplicados,
- conversión de fechas,
- limpieza de texto,
- conservación de URLs y nombres artísticos,
- y exportación del dataset limpio.

In [ ]:
# ============================================
# INSTALACIÓN DE LIBRERÍAS NECESARIAS
# ============================================

%pip install pandas numpy matplotlib seaborn plotly duckdb requests -q

In [ ]:
# ============================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================

import pandas as pd
import numpy as np
import re
import os
import zipfile
import warnings
import requests
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import duckdb

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

# Carpetas locales temporales para ejecutar el notebook
DATA_DIR = Path("data")
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
VISUALIZATIONS_DIR = Path("visualizations")

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías cargadas correctamente.")

## Carga del dataset original de Grammy Awards

Este notebook limpia el dataset original de Grammy Awards para generar la base procesada:

`data/processed/grammy_awards_clean_final.csv`

El archivo original se encuentra en el repositorio dentro de:

`data/raw/Data_spotify.zip`

Para que el notebook pueda ejecutarse en Jupyter Notebook, VS Code o cualquier entorno con conexión a internet, el archivo ZIP se descarga directamente desde GitHub usando su enlace `raw`.



In [ ]:
# ============================================
# DESCARGA DEL ZIP ORIGINAL DESDE GITHUB
# ============================================

RAW_GRAMMY_ZIP_URL = "https://raw.githubusercontent.com/isaacnhf-oss/Spotify-Streaming-Audit-/main/data/raw/Data_spotify.zip"

zip_filename = RAW_DATA_DIR / "Data_spotify.zip"

response = requests.get(RAW_GRAMMY_ZIP_URL)

if response.status_code != 200:
    raise ConnectionError(
        f"No se pudo descargar el archivo ZIP desde GitHub. "
        f"Código de estado: {response.status_code}"
    )

with open(zip_filename, "wb") as file:
    file.write(response.content)

print("Archivo ZIP descargado correctamente.")
print("Ruta local temporal:", zip_filename)
print("Tamaño del archivo en bytes:", zip_filename.stat().st_size)

In [ ]:
# ============================================
# EXTRAER ARCHIVO ZIP
# ============================================

extract_folder = RAW_DATA_DIR / "grammy_awards_extracted"
extract_folder.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Archivo extraído correctamente.")
print("Archivos encontrados dentro del ZIP:")

for path in extract_folder.rglob("*"):
    print("-", path.name)

## Lectura del dataset

Después de extraer el archivo comprimido, se busca automáticamente el archivo `.csv` dentro del ZIP.

Si el ZIP contiene más de un CSV, el notebook prioriza archivos cuyo nombre incluya `grammy` o `award`. Si no encuentra uno con esos términos, utiliza el primer CSV disponible dentro del ZIP.

In [ ]:
# ============================================
# CARGA DEL DATASET
# ============================================

csv_files = list(extract_folder.rglob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError(
        "El ZIP fue extraído correctamente, pero no se encontró ningún archivo CSV dentro."
    )

grammy_candidates = [
    file for file in csv_files
    if "grammy" in file.name.lower() or "award" in file.name.lower()
]

if len(grammy_candidates) > 0:
    csv_path = grammy_candidates[0]
else:
    csv_path = csv_files[0]

print("CSV seleccionado para limpieza:")
print(csv_path)

df_grammy_raw = pd.read_csv(csv_path)

print("Dataset cargado correctamente.")
print("Dimensiones originales:", df_grammy_raw.shape)

display(df_grammy_raw.head())

## Exploración inicial

Revisamos la estructura general del dataset:

- número de filas y columnas,
- tipos de datos,
- nombres de columnas,
- valores nulos,
- y posibles duplicados.

Esta etapa permite entender la calidad inicial de la base antes de limpiarla.

In [ ]:
df_grammy_raw.shape

In [ ]:
df_grammy_raw.info()

In [ ]:
df_grammy_raw.columns

In [ ]:
df_grammy_raw.isnull().sum()

In [ ]:
df_grammy_raw.duplicated().sum()

## Limpieza de nombres de columnas

Para trabajar de forma ordenada, transformamos los nombres de columnas a formato `snake_case`.

Esto significa:

- todo en minúsculas,
- sin espacios,
- sin caracteres especiales,
- y usando guiones bajos `_`.

Importante: esta transformación se aplica solo a los nombres de columnas, no a los valores internos del dataset.

In [ ]:
# ============================================
# FUNCIÓN PARA CONVERTIR NOMBRES DE COLUMNAS A SNAKE_CASE
# ============================================

def to_snake_case(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name

In [ ]:
df_grammy = df_grammy_raw.copy()

df_grammy.columns = [
    to_snake_case(col)
    for col in df_grammy.columns
]

df_grammy.columns

## Revisión de duplicados

Eliminamos filas duplicadas para evitar contar registros repetidos en análisis posteriores.

In [ ]:
duplicados_antes = df_grammy.duplicated().sum()

print("Duplicados antes de limpiar:", duplicados_antes)

In [ ]:
df_grammy = df_grammy.drop_duplicates()

duplicados_despues = df_grammy.duplicated().sum()

print("Duplicados después de limpiar:", duplicados_despues)
print("Dimensiones después de eliminar duplicados:", df_grammy.shape)

## Limpieza de columnas de texto

Las columnas de texto se limpian eliminando:

- espacios innecesarios,
- dobles espacios,
- inconsistencias de formato.

No convertiremos todos los valores a `snake_case`, ya que:

- nombres de artistas perderían legibilidad,
- URLs podrían dañarse,
- categorías y títulos perderían claridad,
- y ciertos nombres artísticos especiales perderían significado.


In [ ]:
# ============================================
# IDENTIFICAR COLUMNAS DE FECHA Y TEXTO
# ============================================

date_columns = [
    col for col in df_grammy.columns
    if "date" in col or "published" in col or "updated" in col
]

text_columns = [
    col for col in df_grammy.select_dtypes(include="object").columns
    if col not in date_columns
]

print("Columnas de fecha:", date_columns)
print("Columnas de texto:", list(text_columns))

In [ ]:
# ============================================
# LIMPIEZA DE TEXTO SIN DAÑAR NOMBRES NI URLS
# ============================================

for col in text_columns:

    df_grammy[col] = (
        df_grammy[col]
        .where(df_grammy[col].notna(), pd.NA)
    )

    df_grammy[col] = (
        df_grammy[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Limpieza de texto completada correctamente.")

df_grammy[text_columns].head()

## Conversión de fechas

Las columnas relacionadas con fechas se convierten a formato datetime para facilitar análisis temporales posteriores.

In [ ]:
for col in date_columns:
    df_grammy[col] = pd.to_datetime(
        df_grammy[col],
        errors="coerce",
        utc=True
    )

df_grammy[date_columns].head()

## Revisión final de valores nulos

Después de las transformaciones, revisamos nuevamente los valores nulos.

Esto permite detectar si alguna conversión generó valores faltantes.

In [ ]:
null_summary = (
    df_grammy
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

null_summary

## Validación de calidad de datos

Antes de finalizar el proceso de limpieza, verificamos:

- estructura final del dataset,
- tipos de datos,
- dimensiones,
- nombres de columnas,
- integridad de URLs,
- y preservación de nombres artísticos.

Esta validación permite asegurar que la base se encuentra lista para análisis posteriores.

In [ ]:
df_grammy.head()

In [ ]:
df_grammy.info()

In [ ]:
print("Dimensiones finales del dataset limpio:")
print(df_grammy.shape)

In [ ]:
df_grammy.columns

In [ ]:
# ============================================
# VALIDACIÓN DE COLUMNAS CLAVE
# ============================================

columnas_validacion = [
    col for col in ["artist", "category", "title", "img"]
    if col in df_grammy.columns
]

display(df_grammy[columnas_validacion].head())

## Exportación del dataset limpio

Después de validar la estructura final, se exporta el dataset limpio en:

`data/processed/grammy_awards_clean_final.csv`

Este archivo será utilizado posteriormente por el notebook principal para integrar Grammy Awards con Spotify.

In [ ]:
# ============================================
# EXPORTACIÓN DEL DATASET LIMPIO
# ============================================

output_path = PROCESSED_DATA_DIR / "grammy_awards_clean_final.csv"

df_grammy.to_csv(output_path, index=False, encoding="utf-8")

print("Dataset limpio exportado correctamente.")
print("Ruta de salida:", output_path)
print("Dimensiones exportadas:", df_grammy.shape)

## Conclusiones de la limpieza de datos

Después del proceso de limpieza y preparación del dataset de Grammy Awards, se obtuvieron las siguientes observaciones:

### Hallazgos principales

- Los nombres de columnas fueron estandarizados a formato `snake_case`.
- Los valores de texto fueron limpiados eliminando espacios innecesarios y dobles espacios.
- Las URLs, nombres artísticos, títulos y categorías conservaron su legibilidad original.
- Se revisaron y eliminaron registros duplicados en caso de existir.
- Las columnas relacionadas con fechas fueron convertidas a formato datetime cuando correspondía.
- El dataset quedó preparado para integrarse con Spotify en el notebook principal.

### Resultado final

La base limpia se exportó como:

`data/processed/grammy_awards_clean_final.csv`

Este archivo puede utilizarse para:

- análisis de artistas y categorías,
- cruces con el dataset principal de Spotify,
- análisis de prestigio musical,
- dashboards en Tableau,
- y análisis avanzados del proyecto.

In [ ]:
# ============================================
# RESUMEN FINAL DE LIMPIEZA
# ============================================

print("=" * 70)
print("RESUMEN FINAL DE LIMPIEZA — GRAMMY AWARDS")
print("=" * 70)

print(f"Filas originales: {df_grammy_raw.shape[0]}")
print(f"Columnas originales: {df_grammy_raw.shape[1]}")
print(f"Filas finales: {df_grammy.shape[0]}")
print(f"Columnas finales: {df_grammy.shape[1]}")

if "duplicados_antes" in globals() and "duplicados_despues" in globals():
    print(f"Duplicados eliminados: {duplicados_antes - duplicados_despues}")

print("-" * 70)
print("Archivo limpio exportado:")
print(output_path)

print("=" * 70)
print("Dataset preparado correctamente para análisis e integración con Spotify.")
print("=" * 70)